# Misinformation Homophily Agent Based Model

**Papers:**
1. Yavaş & Yücel (2013) - "Impact of Homophily on Diffusion Dynamics Over Social Networks"
2. Furutani et al. (2023) - "Analysis of Homophily Effects on Information Diffusion on Social Networks"
3. Liu et al. (2019) - "Homophily on Social Networks Changes Evolutionary Advantage in Competitive Information Diffusion"
4. Watts & Strogatz (1998) - "Collective Dynamics of 'Small-World' Networks"

---

## 1. Background

Research question: Does ingroup bias slow down the spread of misinformation across a population and at what bias level does it stay confined to a single group?

This model combines two course concepts:
- **Spreading model (SIS):** agents cycle between Susceptible and Infected - misinformation can be forgotten and re-adopted (Liu et al., 2019)
- **Group dynamics:** agents belong to social groups and are less likely to accept information from out-group neighbors, controlled by the bias parameter β

Two network types are compared:
- **Random (Erdős–Rényi):** each pair of agents shares the same edge probability
- **Small-World (Watts–Strogatz):** high local clustering with short path lengths, reflecting realistic social structures

The bias parameter β ∈ [0, 1] is the key independent variable:
- β = 0: no group bias - out-group information is accepted as easily as in-group information
- β = 1: complete rejection - cross-group misinformation cannot transmit

## 2. Imports & Setup

In [ ]:
import mesa
import pandas as pd
from mesa.space import NetworkGrid
from mesa.datacollection import DataCollector
import networkx as nx
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt

print(f"Mesa version: {mesa.__version__}")

# 3. Agent

Each agent has a group identity (0 or 1) and a status: **S** (Susceptible) or **I** (Infected/spreading).

Transmission rule from an infected agent to a susceptible neighbor:

$$P(\text{infection}) = \begin{cases} p_{\text{base}} & \text{same group}\\ p_{\text{base}} \times (1 - \beta) & \text{different group} \end{cases}$$

Infected agents recover back to S with probability `recovery_prob` per step (SIS: they can be re-infected), reflecting that individuals can forget or stop spreading misinformation<br>
(Liu et al., 2019).

In [ ]:
class HomophilyAgent(mesa.Agent):
    def __init__(self, model: MisinformationModel, group: int):
        super().__init__(model)
        self.group = group # 0 or 1
        self.status = "S" # mark as susceptible individual

    def step(self):
        if self.status != "I":
            return

        neighbors = self.model.grid.get_neighbors(self.pos, include_center=False)
        for neighbor in neighbors:
            if neighbor.status == "S":
                p = self.model.base_prob
                # it is less likely for other groups to get infected
                if neighbor.group != self.group:
                    p *= (1 - self.model.bias)
                # randomly infect neighbors
                if self.random.random() < p:
                    neighbor.status = "I"

        # recover (SIS: back to susceptible)
        if self.random.random() < self.model.recovery_prob:
            self.status = "S"

# 4. Misinformation Model

Two network types:
- **`random`** - Erdős–Rényi: each pair of agents is connected with the same probability, no inherent group structure
- **`small_world`** - Watts–Strogatz: high local clustering with short average path lengths

Group identity is assigned round-robin by node index (`group = i % num_groups`), so network structure and bias parameter β are fully independent.<br>
Misinformation starts in 5% of group 0 agents.

In [ ]:
class MisinformationModel(mesa.Model):
    def __init__(self, n=200, num_groups=2, bias=0.5, base_prob=0.1, recovery_prob=0.05, network_type="random", seed: int | None = None):
        super().__init__(rng=seed)

        self.n = n
        self.num_groups = num_groups
        self.bias = bias
        self.base_prob = base_prob
        self.recovery_prob = recovery_prob
        self.datacollector = DataCollector(
            model_reporters={
                "Susceptible":  lambda m: sum(1 for a in m.agents if a.status == "S") / m.n,            # percentage of susceptible agents
                "Infected":     lambda m: sum(1 for a in m.agents if a.status == "I") / m.n,            # percentage of infected agents
                "Infected_G0":  lambda m: sum(1 for a in m.agents if a.status == "I" and a.group == 0), # number of infected in G0
                "Infected_G1":  lambda m: sum(1 for a in m.agents if a.status == "I" and a.group == 1), # number of infected in G1
            }
        )
        self.datacollector.collect(self)

        if network_type == "random":
            G = nx.erdos_renyi_graph(n, 0.06, seed=seed)
        elif network_type == "small_world":
            G = nx.watts_strogatz_graph(n, k=6, p=0.1, seed=seed)
        else:
            raise ValueError(f"Unknown network_type: {network_type}")

        self.grid = NetworkGrid(G)

        # group agents
        for i, node in enumerate(G.nodes()):
            group = i % num_groups # 0 or 1
            agent = HomophilyAgent(self, group)
            self.grid.place_agent(agent, node)

        # initial agent infection
        # misinformation originates in group0
        group0 = [a for a in self.agents if a.group == 0]
        n_initial = max(1, int(len(group0) * 0.05)) # infect 5% of group size
        for agent in self.random.sample(group0, n_initial):
            agent.status = "I" # mark as infected individual

    def step(self):
         # randomly call step for an agent
        self.agents.shuffle_do("step")
        self.datacollector.collect(self)

# 5. Parameter Sweep

Bias β is swept from 0 to 1 in 21 steps, with 30 runs per value.<br>
A run is counted as "confined" if less than 5% of out-group agents were ever infected.<br>
Tipping point = smallest tested β where at least 50% of runs are confined (random network only).

In [ ]:
BIAS_VALUES = np.round(np.linspace(0, 1, 21), 2)
ITERATIONS = 30
STEPS_PER_RUN = 120
OUTBREAK_THRESHOLD = 0.05

records = []

for network_type in ["random", "small_world"]:
    for i, bias in enumerate(tqdm(BIAS_VALUES)):
        for run in range(ITERATIONS):
            # deterministic seed per (bias_index, run) for reproducibility - from 1 to max int
            seed = int(np.random.default_rng([i, run]).integers(0, 2**31 - 1))
            model = MisinformationModel(n=200, bias=float(bias), base_prob=0.1, network_type=network_type, seed=seed)

            outbreak_step = np.nan
            group_size = model.n // model.num_groups

            for step in range(STEPS_PER_RUN):
                model.step()
                outgroup_infected = model.datacollector.model_vars["Infected_G1"][-1] / group_size

                # track first step where outgroup exceeds OUTBREAK_THRESHOLD (5%) threshold
                if np.isnan(outbreak_step) and outgroup_infected >= OUTBREAK_THRESHOLD:
                    outbreak_step = step

            data = model.datacollector.get_model_vars_dataframe()
            # final infected fraction in outgroup at end of run
            final_outgroup = data["Infected_G1"].iloc[-1] / group_size
            # peak infected fraction in source group over entire run
            source_peak = data["Infected_G0"].max() / group_size

            records.append({
                "Bias":                         float(bias),
                "Run":                          run,
                "Final_Outgroup_Frac":          final_outgroup,
                "Source_Peak_Infected_Frac":    source_peak,
                "Is_Confined":                  int(final_outgroup <= OUTBREAK_THRESHOLD),
                "Time_To_Outbreak":             outbreak_step,
                "Network":                      network_type,
            })

sweep_df = pd.DataFrame(records)

summary = (
    sweep_df
    .groupby(["Bias", "Network"])
    .agg(
        Mean_Final_Outgroup_Frac =  ("Final_Outgroup_Frac", "mean"),
        SD_Final_Outgroup_Frac   =  ("Final_Outgroup_Frac", "std"),
        Mean_Source_Peak         =  ("Source_Peak_Infected_Frac", "mean"),
        P_Confinement            =  ("Is_Confined", "mean"),
        Mean_Time_To_Outbreak    =  ("Time_To_Outbreak", "mean"),
        No_Outbreak_Frac         =  ("Time_To_Outbreak", lambda s: s.isna().mean()),
    )
    .reset_index()
)

# tipping point: smallest bias where >= 50% of runs stayed confined (random only)
df_random = summary[summary["Network"] == "random"].sort_values("Bias").reset_index(drop=True)
above = df_random[df_random["P_Confinement"] >= 0.50]

if above.empty:
    tipping_point = np.nan
    tipping_message = "Confinement did not reach 50% in the tested bias range."
else:
    first_idx = above.index[0]
    first_bias = df_random.loc[first_idx, "Bias"]

    if first_idx == 0:
        tipping_point = first_bias
        tipping_message = f"At β = {first_bias:.2f}, confinement is already at least 50%."
    else:
        x0, y0 = df_random.loc[first_idx - 1, "Bias"], df_random.loc[first_idx - 1, "P_Confinement"]
        x1, y1 = df_random.loc[first_idx, "Bias"], df_random.loc[first_idx, "P_Confinement"]

        tipping_point = x0 + (0.50 - y0) * (x1 - x0) / (y1 - y0) if y1 != y0 else first_bias
        tipping_message = (
            f"Estimated 50% confinement tipping point: β ≈ {tipping_point:.2f}. "
            f"The first tested bias level above 50% confinement is β = {first_bias:.2f}."
        )

print(tipping_message)
summary.head()

# 6. Plot

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for network_type, style in [("random", "o-"), ("small_world", "s--")]:
    s = summary[summary["Network"] == network_type]
    valid_time = s.dropna(subset=["Mean_Time_To_Outbreak"])

    axes[0].plot(s["Bias"], s["Mean_Final_Outgroup_Frac"], style, label=f"Out-groups ({network_type})")
    axes[0].plot(s["Bias"], s["Mean_Source_Peak"], style, label=f"Source peak ({network_type})")

    axes[1].plot(s["Bias"], s["P_Confinement"], style, label=network_type)

    axes[2].plot(valid_time["Bias"], valid_time["Mean_Time_To_Outbreak"], style, label=f"Time to outbreak ({network_type})")
    axes[2].plot(s["Bias"], s["No_Outbreak_Frac"] * STEPS_PER_RUN, style, label=f"No-outbreak frac ({network_type})")

axes[0].axhline(OUTBREAK_THRESHOLD, linestyle="--", alpha=0.7, label="5% outbreak threshold")
axes[0].set_title("Spread extent")
axes[0].set_xlabel("In-group bias β")
axes[0].set_ylabel("Fraction of agents")
axes[0].set_ylim(0, 1.05)
axes[0].grid(alpha=0.3)
axes[0].legend(fontsize=8)

axes[1].axhline(0.5, linestyle="--", alpha=0.7)
axes[1].set_title("Probability of confinement")
axes[1].set_xlabel("In-group bias β")
axes[1].set_ylabel("P(confined)")
axes[1].set_ylim(0, 1.05)
axes[1].grid(alpha=0.3)
axes[1].legend(fontsize=8)

axes[2].set_title("Spreading speed / delay")
axes[2].set_xlabel("In-group bias β")
axes[2].set_ylabel("Simulation steps")
axes[2].set_ylim(0, STEPS_PER_RUN + 5)
axes[2].grid(alpha=0.3)
axes[2].legend(fontsize=8)

fig.suptitle("Effect of in-group bias on misinformation spread")
plt.tight_layout()
# plt.savefig("sweep_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

summary.round(3)

# 7. Summary